In [2]:
import pandas as pd 
import numpy as np
import pickle as pk 
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.options.display.max_columns = 100
pd.options.display.max_rows = 60


# Preface: Evaluating Selection Bias on eICU Prediction Tasks
In machine learning, we often assume that the data we train on is representative of the population we care about. However, in real-world settings such as healthcare, this assumption frequently does not hold. Selection bias arises when the data we observe is not a random sample of the target population, which can lead to models that perform poorly or unpredictably when deployed.

In this notebook, you will:
1. Train a simple prediction model on real-world ICU data
2. Evaluate how well the model performs across different subsets of the data
3. Explore how differences between training and target populations can affect model performance
4. Implement a very simple (but perhaps unrealistic) solution that corrects for selection bias 

We will begin to build intuition for how selection bias can impact ML models, and how models that perform well on one dataset may fail to generalize to others. Let’s get started!

---

# Load the data

In this notebook, our task will be to predict Y= `mortality_48h`. 
That is, using data from the first 24 hours of hospitalization, predict whether the patient will die within 48 hours.


In [4]:
patient_agg = pd.read_parquet('../../data/clean_dataset.parquet')
feature_sets = pk.load(open('../../data/feature_names.pkl','rb'))
patient_agg

,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep","admissiondx_Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Arteriovenous malformation, surgery for","admissiondx_Arthritis, rheumatoid","admissiondx_Arthritis, septic",...,calcium,cd 4,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,128919,Female,70,Caucasian,59,2015,1,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,101.500000,NaN,2.125000,NaN,0.500000,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,12.500000,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,211.000000,70.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.000000,3.35,NaN,7.10,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN
1,128927,Female,52,Caucasian,60,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,8.100000,NaN,107.750000,NaN,0.700000,NaN,3.000000,234.0,NaN,NaN,NaN,74.5,NaN,NaN,NaN,NaN,NaN,45.000000,1.900,7.000000,NaN,NaN,NaN,NaN,NaN,273.000000,45.0,3.750000,NaN,NaN,NaN,NaN,NaN,NaN,2.3,NaN,NaN,144.500000,0.40,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,Male,68,Caucasian,73,2015,0,>= 500,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.200000,NaN,103.500000,NaN,2.725000,0.1,0.500000,NaN,NaN,NaN,NaN,151.5,NaN,NaN,NaN,1.666667,NaN,3.500000,1.200,4.500000,NaN,NaN,NaN,NaN,NaN,265.500000,91.5,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.500000,0.40,NaN,7.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.018
3,128943,Male,71,Caucasian,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,98.000000,NaN,0.865000,NaN,0.000000,NaN,NaN,NaN,NaN,153.0,NaN,NaN,NaN,0.800000,NaN,3.00

In [5]:
# This distinction of numerical (continuous), binary, and categorical columns may be useful for you 
num_cols = feature_sets['labs'].tolist() + feature_sets['vitals'].tolist() + ['age', 'admissionheight', 'admissionweight']
bin_cols = feature_sets['icd10_before24h'].tolist() + feature_sets['admissiondx'].tolist()
cat_cols = ['hospital_region', 'ethnicity', 'gender', 'hospital_numbedscategory', 'hospitaldischargeyear', 'hospitalid']

# For privacy I think, the dataset sets the age of everyone over 89 as the string '>89'. We set back to a continuous number 
if 'age' in patient_agg.columns: 
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)


In [6]:
Y_label = 'mortality_at48h'
X_covariate_names = num_cols + bin_cols + cat_cols

In [7]:
X = patient_agg[X_covariate_names].copy()
Y = patient_agg[Y_label].copy()
X

,24 h urine protein,24 h urine urea nitrogen,ALT (SGPT),ANF/ANA,AST (SGOT),Acetaminophen,Amikacin peak,Amikacin random,Amikacin trough,BNP,BUN,Base Deficit,Base Excess,CPK,CPKMB,CPKMB INDEX,CRP,CRPhs,Carbamazepine,Carboxyhemoglobin,Clostridium difficile toxin A+B,Cyclosporin,Device,Digoxin,ESR,Fe,Fe/TIBC Ratio,Ferritin,FiO2,Gentamicin peak,Gentamicin random,Gentamicin trough,HCO3,HDL,HIV 1&2 AB,HSV 1&2 IgG AB,HSV 1&2 IgG AB titer,Hct,Hgb,LDH,LDL,LPM O2,Legionella pneumophila Ab,Lidocaine,Lithium,MCH,MCHC,MCV,MPV,Methemoglobin,...,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/arteriovenous malformation,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_TURP, transurethral prostate resection for cancer","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for bronchopleural fistula,admissiondx_Thoracotomy for esophageal cancer,admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for lung reduction,admissiondx_Thoracotomy for other malignancy in chest,admissiondx_Thoracotomy for other reasons,admissiondx_Thoracotomy for pleural disease,admissiondx_Thoracotomy for thoracic/respiratory infection,admissiondx_Thrombectomy (with general anesthesia),admissiondx_Thrombectomy (without general anesthesia),admissiondx_Thrombocytopenia,"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial",admissiondx_Thyroid neoplasm,admissiondx_Thyroidectomy,admissiondx_Thyroidectomy and Parathyroidectomy,"admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Transplant, other","admissiondx_Trauma medical, other","admissiondx_Trauma surgery, other",admissiondx_Tricuspid valve surgery,"admissiondx_Tumor removal, intracardiac","admissiondx_Ulcer disease, peptic","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Vasculitis,admissiondx_Vena cava clipping,admissiondx_Vena cava filter insertion,admissiondx_Ventricular Septal Defect (VSD) Repair,admissiondx_Ventriculostomy,admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_Whipple-surgery for pancreatic cancer,admissiondx_nan,hospital_region,ethnicity,gender,hospital_numbedscategory,hospitaldischargeyear,hospitalid
0,NaN,NaN,199.0,NaN,468.5,NaN,NaN,NaN,NaN,NaN,26.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.250000,13.150,NaN,NaN,NaN,NaN,NaN,NaN,29.20,32.650000,89.350000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Midwest,Caucasian,Female,<100,2015,59
1,NaN,NaN,52.0,NaN,40.0,NaN,NaN,NaN,NaN,NaN,15.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.600000,15.500,NaN,NaN,NaN,NaN,NaN,NaN,33.70,35.600000,94.800000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Midwest,Caucasian,Female,<100,2015,60
2,NaN,NaN,19.5,NaN,19.5,NaN,NaN,NaN,NaN,NaN,36.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.300000,9.350,NaN,NaN,NaN,NaN,NaN,NaN,26.40,33.050000,80.050000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Midwest,Caucasian,Male,>= 500,2015,73
3,NaN,NaN,18.5,NaN,16.0,NaN,NaN,NaN,NaN,24.0,15.000000,NaN,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,NaN,NaN,34.050000,10.900,NaN,NaN,3.0,NaN,NaN,NaN,27.65,32.050000,86.250000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

# Q1: Investigate selection bias 
Let's see how the distribution of features looks like if our data was actually a biased subsample of what we fully observe. 

In [10]:
# TODO: Using the original patient_agg table, look at how gender, age, ethnicity, mortality_at48h, 
# and overall missingness rates differ across hospital_region. 
# You can also see how these features differ across hospitaldischargeyear or hospitalid.

In [13]:
# TODO: Is this dataset representative of the whole U.S., based on national census counts?
# (You can find gender, age, ethnicity rates online.)
# Why might some groups be over- or under-represented in this ICU dataset? 

In [11]:
# TODO: Reflect: If we only observed data from the Northeast 
# (as is the case with the well-known MIMIC dataset!), what might be 
# the implications for model performance on a hospital in Stanford? 

# Q2: Data preprocessing 
Right now dim(X) is huge with 1386 columns, highly sparse, and contains many missing features. 
We will need to do some preprocessing before we can train a model.  

In [ ]:
na_drop_threshold = 0.95
bin_thresh = 0.01

mask = X.isna().mean() > na_drop_threshold
print(f"Dropping columns whose missing rate >{na_drop_threshold}: ", X.columns[mask])
X = X.drop(columns=X.columns[mask])
mask = ~X[bin_cols].mean().between(bin_thresh, 1 - bin_thresh)
print(f"Dropping binary columns whose mean before imputation are not in [{bin_thresh}, 1-{bin_thresh}]: ", X[bin_cols].columns[mask])
X = X.drop(columns=X[bin_cols].columns[mask])
mask = ~X[bin_cols].mean().between(bin_thresh, 1 - bin_thresh)
print(f"Dropping binary columns whose mean before imputation are not in [{bin_thresh}, 1-{bin_thresh}]: ", X[cat_cols].columns[mask])
X = X.drop(columns=X[cat_cols].columns[mask])

num_cols = np.intersect1d(X.columns, num_cols).tolist()
cat_cols = np.intersect1d(X.columns, cat_cols).tolist()
bin_cols = np.intersect1d(X.columns, bin_cols).tolist()

X

In [ ]:
# TODO: Choose your own way to preprocess X such that it no longer contains missing values and has reduced dimensionalitiy.
# You are free to use the above way but you can also do something else. 
# The goal is to get a dataset that is more manageable (for NxM matrix, a large M can lead to issues in modeling)
# This can be either by hand-selecting what features are relevant for Y, 
# removing features with low information (ie. high missingness rates, low variance, etc.)

# Q3: Train a naive classifier model to predict mortality using ALL the data 


In [ ]:
### TODO: Use sklearn to build a simple logistic regression model (or a classifier of your choice) to predict Y given X. 
# You can use your solution from week1 notebook. 
## Output the AUROC and AUPRC score on the test set. 
## Hint: I recommend using StandardScaler for normalizing continuous features 

In [ ]:
# TODO: How does the classifier perform, on average, on different hospital regions?

# Q4: Train a classifier on a subset of the data 
Now suppose we only observe data from the `hospital_region` = West. That is, the data is not fully representative of the whole U.S.

In [ ]:
# TODO: Train your classifier on only this data to predict mortality. Evaluate on the other heldout hospital regions. 
# Do these scores differ at all from Q3? 

# Q5: Reweighting to Address Selection Bias

In many real-world settings, the distribution of the training data may differ from the distribution we care about at deployment time. This mismatch is one form of selection bias and can lead to models that do not generalize well.

One simple way to address this is **reweighting**. The solution of reweighting we will use here is as in [Fair Generative Modeling via Weak Supervision](https://arxiv.org/pdf/1910.12008). The general idea is that we reweight the samples in the training data we have (the biased data) during learning our classifier to assign higher importance to the training examples that look more like the target (test) population, and lower importance to those that do not. To understand what training examples "look like the target population", we will train a second classifier to distinguish between the two datasets. 

Specifically, the weights are called the *density ratio* weights (sometimes also *propsensity weights*) between the training and test distribution. For a datapoint in the training set $i$ with data $(Y_i, X_i)$, then its weight is 
$$w_i = \frac{P(S=1 | Y_i, X_i)}{1-P(S=1 | Y_i, X_i)}$$
where the binary variable $S$ indicates the point belongs in the test distribution. For greater intuition of why these weights work, see the paper. 

Suppose the training data is all eICU patients in the `hospital_region` = West, and the test data is the entire dataset. 

1. Train a second classifier with the input $(Y,X)$ to distinguish training data from test data.
2. Use its predicted probabilities to estimate how likely each training point is to belong to the target population.
3. Convert these probabilities into weights using the equation above.
4. Retrain the prediction model for mortality using these weights.

Intuitively, this procedure helps make the training data "look more like" the test distribution.

Example code: 
```

X_combined = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)

# Label: 0 = train, 1 = test (target population)
s = np.concatenate([
    np.zeros(len(X_train)),
    np.ones(len(X_test))
])

# Split for training the selection model
X_s_train, X_s_val, s_train, s_val = train_test_split(
    X_combined, s, test_size=0.3, random_state=42, stratify=s
)

# ----------------------------------------
# Step 2: Train a classifier to predict train vs test
# ----------------------------------------

selection_model = LogisticRegression(max_iter=1000)
selection_model.fit(X_s_train, s_train)

# Evaluate (optional)
s_val_pred = selection_model.predict_proba(X_s_val)[:, 1]
print("Selection model AUC:", roc_auc_score(s_val, s_val_pred))

# ----------------------------------------
# Step 3: Compute weights for training data
# ----------------------------------------

# Predict probability of being from test distribution
p_test = selection_model.predict_proba(X_train)[:, 1]

# Importance weights: w(x) = p(test | x) / p(train | x)
weights = p_test / (1 - p_test)

print("Weight stats:", weights.min(), weights.max(), weights.mean())

# ----------------------------------------
```

In [ ]:
# TODO: Try this out for yourself and evaluate the performance of 
# the model on different hospital regions. Did it improve? 

In [ ]:
# TODO: Reflect on the assumptions we made to compute these weights. For one, we required access to the 
# full test data (the entire eICU dataset) in order to make a more generalizable model.
# Is this a reasonable assumption? 

# Q6: Different types of dataset shift shift 
Domain adaptation will frequently use the terms *covariate shift* and *label shift*. In this question, we'll think if this is appropriate for our setting. 

In [ ]:
## TODO: Reflect: In our problem setting with our X and Y, are we in a case of covariate (X only) shift, 
# label (Y only) shift, concept shift, or something else? What does this mean 
# if we want to use a method that only works on covariate shift? 